In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
raw_path = '/content/drive/MyDrive/AI_PriceOptima_Data/retail_store_inventory.csv'
updated_path = '/content/drive/MyDrive/AI_PriceOptima_Data/updated_retail_inventory.csv'
clean_path = '/content/drive/MyDrive/AI_PriceOptima_Data/clean_dynamic_pricing_dataset.csv'
feature_path = '/content/drive/MyDrive/AI_PriceOptima_Data/feature_engineered_dataset.csv'

In [3]:
import pandas as pd

# Load updated dataset
df = pd.read_csv(updated_path)

# Check shape and first rows
print(df.shape)
df.head()

(73100, 17)


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,visitors,cost
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn,202,23.450
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn,448,44.107
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer,370,19.593
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn,206,22.904
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,171,51.548


In [4]:
# Check datatype of each column
df.dtypes

,0
Date,object
Store ID,object
Product ID,object
Category,object
Region,object
Inventory Level,int64
Units Sold,int64
Units Ordered,int64
Demand Forecast,float64
Price,float64


In [5]:
# Convert Date column into datetime format
df["Date"] = pd.to_datetime(df["Date"])

# Check datatype again
df.dtypes

,0
Date,datetime64[ns]
Store ID,object
Product ID,object
Category,object
Region,object
Inventory Level,int64
Units Sold,int64
Units Ordered,int64
Demand Forecast,float64
Price,float64


In [6]:
# Check missing values in every column
df.isnull().sum()

,0
Date,0
Store ID,0
Product ID,0
Category,0
Region,0
Inventory Level,0
Units Sold,0
Units Ordered,0
Demand Forecast,0
Price,0


In [7]:
# Check duplicate rows
df.duplicated().sum()

np.int64(0)

In [8]:
# Convert column names to lowercase and snake_case
df.columns = df.columns.str.lower().str.replace(" ", "_")

# Check updated column names
print(df.columns)

Index(['date', 'store_id', 'product_id', 'category', 'region',
       'inventory_level', 'units_sold', 'units_ordered', 'demand_forecast',
       'price', 'discount', 'weather_condition', 'holiday/promotion',
       'competitor_pricing', 'seasonality', 'visitors', 'cost'],
      dtype='object')


In [9]:
df.columns = df.columns.str.replace("/", "_")
print(df.columns)

Index(['date', 'store_id', 'product_id', 'category', 'region',
       'inventory_level', 'units_sold', 'units_ordered', 'demand_forecast',
       'price', 'discount', 'weather_condition', 'holiday_promotion',
       'competitor_pricing', 'seasonality', 'visitors', 'cost'],
      dtype='object')


In [10]:
# Check negative inventory values
df[df["inventory_level"] < 0]
df = df[df["inventory_level"] >= 0]

In [11]:
# Check rows where units sold exceeds inventory
df[df["units_sold"] > df["inventory_level"]]
df = df[df["units_sold"] <= df["inventory_level"]]

In [12]:
# Check invalid price values
df[df["price"] <= 0]
df = df[df["cost"] <= df["price"]]

In [13]:
# Check rows where cost exceeds price
df[df["cost"] > df["price"]]
df = df[df["cost"] <= df["price"]]

In [14]:
# Check invalid discount values
df[(df["discount"] < 0) | (df["discount"] > 1)]
df["discount"] = df["discount"].clip(0, 1)

In [15]:
# Check invalid visitors
df[df["visitors"] < 0]
df = df[df["visitors"] >= 0]

In [16]:
# Clean category names
df["category"] = df["category"].str.strip().str.title()

# Check sample
df["category"].unique()[:10]

array(['Groceries', 'Toys', 'Electronics', 'Furniture', 'Clothing'],
      dtype=object)

In [17]:
# Clean weather condition names
df["weather_condition"] = df["weather_condition"].str.title()

# Check sample
df["weather_condition"].unique()

array(['Rainy', 'Sunny', 'Cloudy', 'Snowy'], dtype=object)

In [18]:
# Calculate IQR for price
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)
IQR = Q3 - Q1

# Define lower and upper limits
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Remove outliers
df = df[(df["price"] >= lower) & (df["price"] <= upper)]

# Check shape after removing outliers
print(df.shape)

(73100, 17)


In [19]:
# Save cleaned dataset
df.to_csv(clean_path , index=False)

print("Clean dataset saved successfully")

Clean dataset saved successfully


In [20]:
# Feature Engineering Starts Here

In [21]:
import pandas as pd

# Load cleaned dataset
df = pd.read_csv(clean_path )

# Check first rows
df.head()

,date,store_id,product_id,category,region,inventory_level,units_sold,units_ordered,demand_forecast,price,discount,weather_condition,holiday_promotion,competitor_pricing,seasonality,visitors,cost
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,1,Rainy,0,29.69,Autumn,202,23.450
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,1,Sunny,0,66.16,Autumn,448,44.107
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,1,Sunny,1,31.32,Summer,370,19.593
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,1,Cloudy,1,34.74,Autumn,206,22.904
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer,171,51.548


In [22]:
# Profit Margin
df["profit_margin"] = df["price"] - df["cost"]

# Check result
df[["price", "cost", "profit_margin"]].head()

,price,cost,profit_margin
0,33.50,23.450,10.050
1,63.01,44.107,18.903
2,27.99,19.593,8.397
3,32.72,22.904,9.816
4,73.64,51.548,22.092


In [23]:
# Margin Percentage
df["margin_percent"] = (df["price"] - df["cost"]) / df["price"]

# Check result
df[["margin_percent"]].head()

,margin_percent
0,0.3
1,0.3
2,0.3
3,0.3
4,0.3


In [24]:
# Demand Ratio
df["demand_ratio"] = df["units_sold"] / (df["demand_forecast"] + 1)

# Check
df[["units_sold", "demand_forecast", "demand_ratio"]].head()

,units_sold,demand_forecast,demand_ratio
0,127,135.47,0.930607
1,150,144.04,1.034197
2,65,74.02,0.866436
3,61,62.18,0.965495
4,14,9.26,1.364522


In [25]:

# Inventory Pressure
df["inventory_pressure"] = df["units_sold"] / (df["inventory_level"] + 1)

# Check
df[["inventory_level", "units_sold", "inventory_pressure"]].head()

,inventory_level,units_sold,inventory_pressure
0,231,127,0.547414
1,204,150,0.731707
2,102,65,0.631068
3,469,61,0.129787
4,166,14,0.083832


In [26]:
# Competitor Price Gap
df["competitor_gap"] = df["price"] - df["competitor_pricing"]

# Check
df[["price", "competitor_pricing", "competitor_gap"]].head()

,price,competitor_pricing,competitor_gap
0,33.50,29.69,3.81
1,63.01,66.16,-3.15
2,27.99,31.32,-3.33
3,32.72,34.74,-2.02
4,73.64,68.95,4.69


In [27]:
# Discounted Price
df["discounted_price"] = df["price"] * (1 - df["discount"])

# Check
df[["price", "discount", "discounted_price"]].head()

,price,discount,discounted_price
0,33.50,1,0.00
1,63.01,1,0.00
2,27.99,1,0.00
3,32.72,1,0.00
4,73.64,0,73.64


In [28]:
# Revenue
df["revenue"] = df["units_sold"] * df["discounted_price"]

# Check
df[["units_sold", "discounted_price", "revenue"]].head()

,units_sold,discounted_price,revenue
0,127,0.00,0.00
1,150,0.00,0.00
2,65,0.00,0.00
3,61,0.00,0.00
4,14,73.64,1030.96


In [29]:
# Conversion Rate
df["conversion_rate"] = df["units_sold"] / (df["visitors"] + 1)

# Check
df[["units_sold", "visitors", "conversion_rate"]].head()

,units_sold,visitors,conversion_rate
0,127,202,0.625616
1,150,448,0.334076
2,65,370,0.175202
3,61,206,0.294686
4,14,171,0.081395


In [30]:
# Price Demand Ratio
df["price_demand_ratio"] = df["price"] / (df["units_sold"] + 1)

# Check
df[["price", "units_sold", "price_demand_ratio"]].head()

,price,units_sold,price_demand_ratio
0,33.50,127,0.261719
1,63.01,150,0.417285
2,27.99,65,0.424091
3,32.72,61,0.527742
4,73.64,14,4.909333


In [31]:
# Stock Remaining
df["stock_remaining"] = df["inventory_level"] - df["units_sold"]

# Check
df[["inventory_level", "units_sold", "stock_remaining"]].head()

,inventory_level,units_sold,stock_remaining
0,231,127,104
1,204,150,54
2,102,65,37
3,469,61,408
4,166,14,152


In [32]:
# Traffic Intensity
df["traffic_intensity"] = df["visitors"] / (df["inventory_level"] + 1)

# Check
df[["visitors", "inventory_level", "traffic_intensity"]].head()

,visitors,inventory_level,traffic_intensity
0,202,231,0.870690
1,448,204,2.185366
2,370,102,3.592233
3,206,469,0.438298
4,171,166,1.023952


In [33]:
# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Check datatype
df["date"].dtype

dtype('<M8[ns]')

In [34]:
# Day of week
df["day_of_week"] = df["date"].dt.dayofweek

# Check
df[["date", "day_of_week"]].head()

,date,day_of_week
0,2022-01-01,5
1,2022-01-01,5
2,2022-01-01,5
3,2022-01-01,5
4,2022-01-01,5


In [35]:
# Month
df["month"] = df["date"].dt.month

# Check
df[["date", "month"]].head()

,date,month
0,2022-01-01,1
1,2022-01-01,1
2,2022-01-01,1
3,2022-01-01,1
4,2022-01-01,1


In [36]:
# Weekend indicator
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

# Check
df[["day_of_week", "is_weekend"]].head()

,day_of_week,is_weekend
0,5,1
1,5,1
2,5,1
3,5,1
4,5,1


In [37]:
# Save final feature engineered dataset
df.to_csv(feature_path , index=False)

print("Feature engineered dataset saved successfully")

Feature engineered dataset saved successfully


In [38]:
import os

print(os.listdir())

['.config', 'drive', 'sample_data']


In [39]:
from google.colab import files
files.download(feature_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>